# 67 — Final Documentary Forensic Summary

Produces a machine-readable summary that merges contradiction tracking and
documentary evidence strength.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "67_final_forensic_summary"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

patched, _ = safe_read_csv(OUTPUTS / "66_documentary_adjudication_patch" / "site_adjudication_patched.csv")
contradictions, _ = safe_read_csv(OUTPUTS / "57_contradictions_red_team_audit" / "contradiction_register.csv")
docs, _ = safe_read_csv(OUTPUTS / "65_documentary_evidence_fusion" / "documentary_evidence_site_year.csv")

summary = {
    "generated_at_utc": utcnow(),
    "documentary_layer_present": not docs.empty,
    "patched_adjudication_present": not patched.empty,
    "contradiction_count": int(len(contradictions)) if not contradictions.empty else 0,
    "priority_review_sites": [],
    "documentary_supported_sites": [],
}

if not patched.empty:
    summary["priority_review_sites"] = patched.loc[patched["verdict_patched"] == "priority_review", "site_id"].astype(str).tolist()
    summary["documentary_supported_sites"] = patched.loc[patched["verdict_patched"] == "documentary_supported_review", "site_id"].astype(str).tolist()

write_json(out / "final_documentary_forensic_summary.json", summary)

lines = [
    "# Final Documentary Forensic Summary",
    "",
    f"Generated: {summary['generated_at_utc']}",
    "",
    f"- Documentary layer present: {summary['documentary_layer_present']}",
    f"- Patched adjudication present: {summary['patched_adjudication_present']}",
    f"- Contradiction count: {summary['contradiction_count']}",
    f"- Priority review sites: {', '.join(summary['priority_review_sites']) if summary['priority_review_sites'] else 'None'}",
    f"- Documentary supported sites: {', '.join(summary['documentary_supported_sites']) if summary['documentary_supported_sites'] else 'None'}",
]
(out / "final_documentary_forensic_summary.md").write_text("\n".join(lines), encoding="utf-8")

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
add_artifact(manifest, out / "final_documentary_forensic_summary.json")
add_artifact(manifest, out / "final_documentary_forensic_summary.md")
write_json(out / "manifest.json", manifest)

display(pd.DataFrame([summary]))